# Chapter 7 — A stellar stream: the tidal tails of Pal 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/07_pal5_stream.ipynb)

**Goal.** A satellite is a compact overdensity; a disrupted satellite is spread along its orbit into a long, thin
stream. The globular cluster Pal 5 is the classic example: SDSS revealed its two tidal tails (Odenkirchen et al.
2001, https://ui.adsabs.harvard.edu/abs/2001ApJ...548L.165O; Rockosi et al. 2002,
https://ui.adsabs.harvard.edu/abs/2002AJ....124..349R). The same matched filter as for satellites reveals it
(Chapter 4); what changes is the geometry. We rotate to coordinates aligned with the stream, measure its track
and width along it, and judge the significance of each segment against control strips.

**You will learn**
1. to build a great-circle coordinate frame $(\phi_1, \phi_2)$ through a point and along a direction;
2. to measure a stream's track, width and density in bins of $\phi_1$ with a Poisson likelihood that includes the
   survey footprint;
3. to calibrate the significance of stream segments with control strips;
4. to compare the measured track with a published one (`galstreams`, Mateu 2023).

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter                         # smoothing of the field CMD (Hess diagram)
from scipy.optimize import minimize                               # Poisson likelihood fits of the stream profile

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


lvdb = load('lvdb').set_index('key')        # Local Volume Database v1.1.1 (Pace 2025): reference positions, distances
iso = load('isochrones_sdss')               # PARSEC 1.2S, SDSS ugriz: log age 9.9-10.1, [Fe/H] -2.1...-1.0
p5 = lvdb.loc['palomar_05']                 # Pal 5: ra, dec [deg] and distance_modulus [mag] are used below

## 7.1 Selecting stars like Pal 5's

Pal 5 is ~11–12 Gyr old with [Fe/H] ≈ −1.3; its distance modulus is 16.71 (LVDB). We build a CMD mask exactly as in
Chapter 5 (cells ranked by $s/b$, keeping at least half of the cluster's stars), with $b$ from the whole field.

In [ ]:
# CMD grid: colour (g-r)_0 from -0.3 to 1.2 in 0.04-mag bins, magnitude g_0 from 16 to 22.5 in 0.1-mag bins (the colour
# range and the g < 22.5 limit are those of the pal5_sdss extraction). cc, mm: bin centres; CC, MM: their 2-d grid,
# shape (37, 65) = (colour, magnitude).
cb, mb = np.arange(-0.3, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')


def satellite_cmd(mu, logage, feh, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Normalised CMD density of a simple stellar population (as in Chapters 4-5).

    mu: distance modulus [mag]; logage: log10(age / yr); feh: [Fe/H]; err(m): one-band photometric error [mag] at
    magnitude m. Returns s on the CMD grid, shape (37, 65), summing to 1.
    """
    # isochrone of the chosen age and metallicity, evolutionary stages up to 7 (early AGB), ordered by initial mass
    I = iso[np.isclose(iso.logage, logage) & np.isclose(iso.feh, feh) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy()                 # absolute colour and absolute g magnitude
    # 3000 points uniform in arc length along the track in the CMD (uniform in mass would leave fast phases such as the
    # red-giant branch empty); each point is weighted by the stars it represents, the increment of the cumulative IMF
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])
    s = np.linspace(0, arc[-1], 3000)
    pc, pg, w = np.interp(s, arc, c), np.interp(s, arc, g) + mu, np.gradient(np.interp(s, arc, I.int_imf))
    # Gaussian blur of every point. Colour: sqrt(2) x the one-band error (g and r errors in quadrature) plus an
    # intrinsic 0.03 mag; magnitude: the error plus 0.1 mag for the spread in distance and in horizontal-branch
    # luminosity (Chapter 2). err(m) rises from 0.02 mag at the bright end to 0.10 mag at g = 22 (the median SDSS
    # error there is ~0.07 mag).
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    # sum over the 3000 points of weight x 2-d Gaussian on the grid: (3000,) and (3000, 37, 65) -> (37, 65)
    S = np.einsum('k,kij->ij', w, np.exp(-0.5 * ((CC - pc[:, None, None]) / sc[:, None, None])**2
                                         - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_mask(S, B, keep=0.5):
    """Boolean CMD mask (Chapter 5): the cells of highest s/b that maximise sum(s) / sqrt(sum(b)).

    S, B: satellite and field CMD densities on the same grid (each sums to 1). keep: the mask must hold at least this
    fraction of the satellite's stars (half).
    """
    order = np.argsort((S / B).ravel())[::-1]                     # cells ranked by s/b, best first
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])   # signal and background kept by the top k cells
    M = np.zeros(S.size, bool)
    # best k: maximum of cs / sqrt(cbk) among the k with cs >= keep (the others are set to 0 so that argmax skips them)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


cat = load('pal5_sdss')            # SDSS point sources, -0.3 < g-r < 1.2, g < 22.5, RA 222-244 deg, Dec -8...+8 deg
# extinction-corrected g_0 and (g-r)_0 [mag]: the files store raw PSF magnitudes and the SFD98 extinction separately
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
# field CMD density b from all stars of the field (the stream is a tiny fraction of them), smoothed by one cell;
# cells are floored at one star so that s/b stays finite where the field is poorly sampled (Chapter 4)
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
# mask for Pal 5: log age 10.05 (11.2 Gyr), [Fe/H] = -1.3, at the LVDB distance modulus (16.71)
M = cmd_mask(satellite_cmd(p5.distance_modulus, 10.05, -1.3), B)
# CMD cell (i, j) of every star; stars off the grid are rejected explicitly, the clip only keeps their index valid
i, j = np.digitize(c0, cb) - 1, np.digitize(g0, mb) - 1
sel = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm)) & M[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)]
ra, dec = cat.ra.to_numpy()[sel], cat.dec.to_numpy()[sel]          # positions [deg] of the masked stars
print(f'{sel.sum():,} of {len(cat):,} stars in the CMD mask')

## 7.2 A coordinate frame aligned with the stream

Let $\hat v_0$ be the unit vector to Pal 5, and $\hat e$, $\hat n$ the local east and north directions there. A
great circle through Pal 5 at position angle $\alpha$ (north through east) has direction $\hat d = \sin\alpha\,\hat e +
\cos\alpha\,\hat n$ and pole $\hat p = \hat v_0 \times \hat d$. The rotation matrix with rows $(\hat v_0, \hat d, \hat p)$
maps a sky unit vector $\hat v$ to $\hat v' = R\hat v$, and

$$ \phi_1 = {\rm atan2}(v'_y, v'_x), \qquad \phi_2 = \arcsin v'_z ,$$

so Pal 5 is at (0, 0) and the great circle is $\phi_2 = 0$. The inverse is $\hat v = R^{\sf T}\hat v'$. We choose $\alpha$
by maximising the number of masked stars within 0.25° of the great circle at $1.5° < |\phi_1| < 6°$ (avoiding the cluster).

In [ ]:
def unit(ra, dec):
    """Unit vector(s) on the sphere for (ra, dec) in degrees: shape (3,) or (3, N)."""
    ra, dec = np.radians(ra), np.radians(dec)
    return np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def frame(ra0, dec0, alpha):
    """Rotation matrix to stream coordinates through (ra0, dec0) along position angle alpha (deg).

    Rows: v0 (the origin, phi1 = phi2 = 0), d (direction of increasing phi1 at the origin; alpha is measured from
    north through east) and the pole p = v0 x d (phi2 = +90 deg).
    """
    v0 = unit(ra0, dec0)
    a0, d0 = np.radians(ra0), np.radians(dec0)
    e = np.array([-np.sin(a0), np.cos(a0), 0])                                   # local unit vector to the east
    n = np.array([-np.sin(d0) * np.cos(a0), -np.sin(d0) * np.sin(a0), np.cos(d0)])  # local unit vector to the north
    d = np.sin(np.radians(alpha)) * e + np.cos(np.radians(alpha)) * n           # along the great circle
    return np.array([v0, d, np.cross(v0, d)])


def to_stream(R, ra, dec):
    """(ra, dec) [deg] -> stream coordinates (phi1, phi2) [deg], phi1 in (-180, 180]."""
    v = R @ unit(ra, dec)
    # the clip protects arcsin from |v_z| exceeding 1 by rounding
    return np.degrees(np.arctan2(v[1], v[0])), np.degrees(np.arcsin(np.clip(v[2], -1, 1)))


def from_stream(R, phi1, phi2):
    """Inverse of to_stream: (phi1, phi2) [deg] -> (ra, dec) [deg], ra in [0, 360)."""
    v = R.T @ unit(phi1, phi2)                                    # R is orthogonal: its inverse is its transpose
    return np.degrees(np.arctan2(v[1], v[0])) % 360, np.degrees(np.arcsin(np.clip(v[2], -1, 1)))


# Orientation of the frame: for position angles 0-179 deg in 1-deg steps (alpha and alpha + 180 give the same circle),
# count masked stars within 0.25 deg of the great circle at 1.5 < |phi1| < 6 deg, i.e. along the tails, outside the
# cluster, and keep the angle with the largest count.
alphas = np.arange(0, 180, 1.0)
score = []
for a in alphas:
    f1, f2 = to_stream(frame(p5.ra, p5.dec, a), ra, dec)
    score.append(np.sum((np.abs(f2) < 0.25) & (np.abs(f1) > 1.5) & (np.abs(f1) < 6)))
alpha = alphas[np.argmax(score)]
R = frame(p5.ra, p5.dec, alpha)
phi1, phi2 = to_stream(R, ra, dec)                                # stream coordinates [deg] of the masked stars

# round trip on random points: the inverse transformation must recover them
# 1000 points uniform in the field (RA 222-244, Dec -8...+8 deg); rt is the largest angular error [deg] (RA difference
# scaled by cos dec). The tolerance, 1e-6 deg (3.6 mas), is far above double-precision rounding and far below any
# scale in this chapter: a failure means a wrong inverse, not rounding.
rng = np.random.default_rng(7)                                    # fixed seed: the same test points in every run
tra, tdec = rng.uniform(222, 244, 1000), rng.uniform(-8, 8, 1000)
bra, bdec = from_stream(R, *to_stream(R, tra, tdec))
rt = np.max(np.hypot((bra - tra) * np.cos(np.radians(tdec)), bdec - tdec))
print(f'best position angle alpha = {alpha:.0f} deg; round-trip error {rt:.1e} deg')
assert rt < 1e-6

# left: star count near the circle vs alpha; right: masked stars in (phi1, phi2), 0.1-deg bins, grey scale
# saturated at 12 stars per bin so that the faint tails stand out
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(alphas, score)
ax[0].set(xlabel='position angle α of the great circle [deg]', ylabel='masked stars near the circle')
h = ax[1].hist2d(phi1, phi2, [np.arange(-12, 12.01, 0.1), np.arange(-4, 4.01, 0.1)], cmap='Greys', vmax=12)
ax[1].set(xlabel=r'$\phi_1$ [deg]', ylabel=r'$\phi_2$ [deg]', title='masked stars in the stream frame', aspect='equal')
plt.tight_layout()

**Figure 7.1.** Left: the number of CMD-masked stars (the 114,877 of 1,256,881 SDSS point sources of `pal5_sdss`,
extinction-corrected, that fall in the mask for log age 10.05, [Fe/H] = −1.3 at μ = 16.71) within 0.25° of a great
circle through Pal 5 and at $1.5° < |\phi_1| < 6°$, as a function of the circle's position angle α (north through east,
1° steps). Away from the stream direction the count is ~1300; the tails produce a sharp maximum of ~2600 at α = 52°,
followed by a broad shoulder at α ≈ 60°–110°. Right: the masked stars in the stream frame for α = 52° (0.1° × 0.1°
bins, grey scale saturated at 12 stars per bin; $\phi_1$ increases towards the north-east). Pal 5 is the black clump at
(0, 0) and the tails form a thin band near $\phi_2 = 0$, from the footprint edge at $\phi_1 \approx -4°$ to beyond
$+6°$. The dark spot with a white core at (1.8°, 1.4°) is the globular cluster M5, whose crowded centre has no SDSS
point sources. White areas are outside the SDSS footprint (straight diagonal edges); the striped patch at
$\phi_1 \approx -9°$ is a set of isolated SDSS scan stripes. A one-parameter scan of orientation is enough to align the
frame with the stream.

## 7.3 Track, width and significance along the stream

In bins of 1° in $\phi_1$ we model the distribution of masked stars in $\phi_2$ (within ±1.5° of a centre) as a stream
plus background,

$$ \lambda(\phi_2) = \epsilon(\phi_2)\,\Big[ a + b\,\phi_2 + A\, \mathcal N(\phi_2 \mid \mu, \sigma) \Big], $$

where $\epsilon$ is the fraction of each 0.05° sub-bin that SDSS observed (from the coverage grid), and fit the binned
counts with a Poisson likelihood. $TS = 2\Delta\ln\mathcal L$ compares it with the background-only model ($A = 0$).
The parameters $\mu$ and $\sigma$ are free only under the stream model, so TS does not follow $\chi^2$ (Chapter 6).
We calibrate it instead with **control strips**: the same fit centred at $\phi_2 = \pm 3.5°$, where there is no stream.
A segment is detected if its TS exceeds the largest TS of all control fits.

Only strips lying entirely inside the observed area are fitted. In a first version we allowed partly observed strips
and corrected them with the coverage fraction; two control strips that touched the edge of the data then gave
TS ≈ 60–75, because a drop in counts at an edge looks like the side of a stream. Edges and masks are a major
source of false stream detections.

In [ ]:
# Survey footprint: pal5_coverage counts SDSS objects n in 0.1 x 0.1 deg cells of (ra, dec), indexed by (ira, idec).
cov = load('pal5_coverage')
cra, cdec = (cov.ira.to_numpy() + 0.5) * 0.1, (cov.idec.to_numpy() + 0.5) * 0.1   # cell centres [deg]
obs = cov.n.to_numpy() > 20              # observed cell; observed cells hold a median of ~140 objects, so the cut
                                         # rejects only empty or barely covered cells
# centres of all 0.1-deg cells of the field, observed or not: shape (220, 160)
allra, alldec = np.meshgrid(np.arange(222, 244, 0.1) + 0.05, np.arange(-8, 8, 0.1) + 0.05, indexing='ij')
c1, c2 = to_stream(R, cra[obs], cdec[obs])                        # observed cell centres in the stream frame [deg]
a1, a2 = to_stream(R, allra.ravel(), alldec.ravel())              # all cell centres in the stream frame [deg]

f1_edges = np.arange(-8, 8.01, 1.0)                               # 1-deg segments in phi1 from -8 to +8 deg
sub = 0.05                                                        # phi2 sub-bin width [deg], below the stream width
off = np.arange(-1.5, 1.5001, sub)                                # phi2 sub-bin edges relative to the strip centre


def exposure(lo, hi, centre):
    """Observed fraction of each phi2 sub-bin for phi1 in [lo, hi), from the 0.1-deg coverage cells.

    The fraction is (observed cell centres) / (all cell centres) falling in the sub-bin; 0 where no cell centre falls
    (outside the extracted field). Returns an array of 60 values, one per sub-bin of the strip centred at `centre`.
    """
    e = centre + off
    n_obs = np.histogram(c2[(c1 >= lo) & (c1 < hi)], e)[0]
    n_all = np.histogram(a2[(a1 >= lo) & (a1 < hi)], e)[0]
    return np.where(n_all > 0, n_obs / np.maximum(n_all, 1), 0.0)


def fit_strip(lo, hi, centre):
    """Fit stream + linear background to masked stars with phi1 in [lo, hi) and |phi2 - centre| < 1.5.

    Returns None for a strip not fully observed, otherwise TS = 2 Delta ln L (stream vs background only), the stream
    centre mu [deg, absolute phi2], width sigma [deg], number of stream stars A in the segment, and the total count n.
    """
    eps = exposure(lo, hi, centre)
    if eps.min() < 0.9:                                           # only strips fully inside the observed area
        return None                                               # (every sub-bin at least 90% observed)
    x = 0.5 * (off[1:] + off[:-1])                                # sub-bin centres relative to the strip centre [deg]
    n = np.histogram(phi2[(phi1 >= lo) & (phi1 < hi)] - centre, off)[0]   # masked stars per sub-bin

    def nll(q, stream=True):
        """Poisson -ln L (without the constant ln n! term) of the counts n.

        q = (a, b): background [stars per sub-bin] and its slope [per deg]; with stream=True also ln A, mu, s:
        number of stream stars, Gaussian centre and width [deg], relative to the strip centre.
        """
        a, b = q[0], q[1]
        lam = a + b * x                                           # expected background counts per sub-bin
        if stream:
            A, mu, s = np.exp(q[2]), q[3], q[4]                   # A is fitted as ln A, so it stays positive
            # Gaussian holding A stars in total; the factor sub turns a density per degree into counts per sub-bin
            lam = lam + A * sub * np.exp(-0.5 * ((x - mu) / s)**2) / (np.sqrt(2 * np.pi) * s)
        lam = np.maximum(lam * eps, 1e-9)                         # only the observed fraction is counted; the floor
                                                                  # keeps the log finite if a + b x goes negative
        return -np.sum(n * np.log(lam) - lam)

    # starting background: mean count per sub-bin corrected for the mean exposure, at least 0.1 star
    a0 = max(n[eps > 0].mean() / max(eps[eps > 0].mean(), 1e-3), 0.1)
    r0 = minimize(nll, [a0, 0.0], args=(False,), method='Nelder-Mead')   # background-only fit (A = 0), the null model
    best = None
    # The likelihood is multimodal in mu, so the stream fit starts from five centres 0.3 deg apart and keeps the best.
    # Start: background of r0, A = 2 a0, sigma = 0.15 deg. Bounds: a > 0; ln A in (-5, 8), i.e. A from 0.007 to 3000
    # stars; |mu| < 1.2 deg keeps the Gaussian inside the +-1.5 deg strip; sigma from 0.05 deg (one sub-bin) to 0.5 deg.
    for mu0 in (-0.6, -0.3, 0.0, 0.3, 0.6):                       # several starting points for the stream centre
        r = minimize(nll, [*r0.x, np.log(max(a0 * 2, 1)), mu0, 0.15], method='L-BFGS-B',
                     bounds=[(1e-3, None), (None, None), (-5, 8), (-1.2, 1.2), (0.05, 0.5)])
        if best is None or r.fun < best.fun:
            best = r
    return dict(TS=2 * (r0.fun - best.fun), mu=best.x[3] + centre, sigma=best.x[4], A=np.exp(best.x[2]), n=n.sum())


rows = []
for lo, hi in zip(f1_edges[:-1], f1_edges[1:]):
    if lo < 1 and hi > -1:                                        # skip the cluster itself
        continue                                                  # (the two segments with |phi1| < 1 deg)
    # the stream strip (centred on phi2 = 0) and two control strips at phi2 = -3.5 and +3.5 deg, where no stream is
    # expected; a strip that is not fully observed gives None, stored as NaN
    fits = {c: fit_strip(lo, hi, c) for c in (0.0, -3.5, 3.5)}
    # one row per segment; columns '<quantity>_<strip centre>', e.g. TS_0.0, mu_-3.5
    rows.append(dict(phi1=0.5 * (lo + hi), **{f'{k}_{c}': (v[k] if v else np.nan) for c, v in fits.items() for k in ('TS', 'mu', 'sigma', 'A')}))
tr = pd.DataFrame(rows)
ctrl = np.concatenate([tr['TS_-3.5'], tr['TS_3.5']])              # TS of all control fits
thresh = np.nanmax(ctrl)                                          # detection threshold: the largest control TS
tr['detected'] = tr['TS_0.0'] > thresh
print(f'largest control-strip TS: {thresh:.1f}')
print(tr[['phi1', 'TS_0.0', 'mu_0.0', 'sigma_0.0', 'A_0.0', 'detected']].round(2).to_string(index=False))

## 7.4 Comparison with a published track

`galstreams` (Mateu 2023, https://ui.adsabs.harvard.edu/abs/2023MNRAS.520.5225M) collects published stream tracks.
We take the Pal 5 track of Price-Whelan et al. (2019) from a fixed commit of the `galstreams` repository
(BSD-3 licence), transform it into our frame and compare.

In [ ]:
# Pal 5 track of Price-Whelan et al. (2019) in galstreams, pinned to a fixed commit: the comparison is reproducible
GALSTREAMS = ('https://raw.githubusercontent.com/cmateu/galstreams/6a304c05bea7f774390a11082ea127719e99b70a/'
              'galstreams/tracks/track.st.Pal5.pricewhelan2019.ecsv')
gs = pd.read_csv(GALSTREAMS, comment='#')                         # ECSV: the header lines start with '#'
g1, g2 = to_stream(R, gs.ra.to_numpy(), gs.dec.to_numpy())         # published track in our frame [deg]
o = np.argsort(g1)                                                # sort by phi1 for interpolation and plotting
gs_phi2 = np.interp(tr.phi1, g1[o], g2[o])                        # published phi2 at the centre of each 1-deg segment
det = tr[tr.detected]
resid = det['mu_0.0'] - gs_phi2[tr.detected]                      # measured minus published track [deg]
print(f'detected segments: {tr.detected.sum()} ({(det.phi1 < 0).sum()} with phi1 < 0, {(det.phi1 > 0).sum()} with phi1 > 0); '
      f'median |track - galstreams| = {np.median(np.abs(resid)):.2f} deg')

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
# top: masked stars in 0.1 x 0.05 deg bins (grey scale saturated at 10), the published track, the fitted centre +- width
ax[0].hist2d(phi1, phi2, [np.arange(-8, 8.01, 0.1), np.arange(-2, 2.01, 0.05)], cmap='Greys', vmax=10)
ax[0].plot(g1[o], g2[o], c='C1', lw=1, label='galstreams: Price-Whelan et al. (2019)')
ax[0].errorbar(det.phi1, det['mu_0.0'], yerr=det['sigma_0.0'], fmt='o', c='C3', label='this fit: centre ± width')
ax[0].set(ylabel=r'$\phi_2$ [deg]', ylim=(-2, 2), xlim=(-8, 8))
ax[0].legend(fontsize=8)
# middle: TS of the stream strip per segment (red if detected, grey if not), control strips as triangles, threshold
# dashed; symlog axis (linear below 1) so that TS from ~3 to ~80 fit on one plot
ax[1].bar(tr.phi1, tr['TS_0.0'], width=0.8, color=np.where(tr.detected, 'C3', '0.6'), label='stream strip')
ax[1].plot(tr.phi1, tr['TS_-3.5'], 'v', c='C0', label='control strips')
ax[1].plot(tr.phi1, tr['TS_3.5'], '^', c='C0')
ax[1].axhline(thresh, c='C0', ls='--', lw=1)
ax[1].set(ylabel='TS', yscale='symlog')
ax[1].legend(fontsize=8)
# bottom: fitted number of stream stars A in each detected 1-deg segment
ax[2].errorbar(det.phi1, det['A_0.0'], fmt='o', c='k')
ax[2].set(xlabel=r'$\phi_1$ [deg]', ylabel='stream stars per degree')
plt.tight_layout()
# Tests of this chapter: the stream is detected (TS above the control threshold) in at least two disjoint 1-deg segments
# on each side of the cluster, and the median |measured - published track| is below 0.2 deg, comparable to the fitted
# widths (0.08-0.24 deg); the measured value is 0.02 deg.
assert (det.phi1 < 0).sum() >= 2 and (det.phi1 > 0).sum() >= 2
assert np.median(np.abs(resid)) < 0.2

**Figure 7.2.** Top: the masked stars in the stream frame (0.1° in $\phi_1$ × 0.05° in $\phi_2$, grey scale saturated
at 10 stars per bin); red points with bars: the fitted stream centre μ ± Gaussian width σ in the six detected 1°
segments; orange line: the Pal 5 track of Price-Whelan et al. (2019) from `galstreams`, transformed into our frame (it
continues to $\phi_1 \approx -6°$, beyond the SDSS data). Middle: TS of the stream + linear-background fit to the
strip $|\phi_2| < 1.5°$ in each 1° segment (bars, symmetric-log axis; red: detected, grey: not detected), and of the
control strips centred at $\phi_2 = -3.5°$ (▼) and $+3.5°$ (▲); the dashed line is the largest control TS, 8.7, the
detection threshold. Segments with no bar are not fitted: $|\phi_1| < 1°$ (the cluster) and strips not fully observed.
Bottom: fitted number of stream stars A in each detected 1° segment (no uncertainties shown). Six segments are
detected, two at $\phi_1 < 0$ (TS = 31 and 82) and four at $\phi_1 > 0$ (TS = 77, 39, 18 and 26); the segment at
$\phi_1 = 7.5°$ (TS = 7.5) falls just below the threshold. The tails follow the published track (median offset 0.02°),
are only 0.08–0.24° wide (30–90 pc at 22 kpc), and thin out away from the cluster, from ~140–160 stars per degree at
$|\phi_1| < 3°$ to ~70 at $\phi_1 = 6.5°$. Strips at $\phi_1 < -3°$ reach beyond the SDSS footprint (the tail itself
is visible to $\phi_1 \approx -4°$), and strips containing a coverage hole are not fitted either: the empty core of M5
(segment at $\phi_1 = 1.5°$) and a single empty 0.1° cell at (RA, Dec) = (232.15°, 1.85°), right on the track (segment
at $\phi_1 = 3.5°$).

A few caveats. Our CMD mask uses one distance for the whole stream; in reality the tails span a range of distances,
so the mask loses stars at the ends. The great circle is only an approximation of the curved track (the fit absorbs
it through μ). And with only two control strips per segment, the threshold is a rough calibration: a stricter one
uses many control strips, or injected streams (Chapter 8).

## Summary

- A great-circle frame through a stream turns a curved, inclined structure into a nearly horizontal band.
- Binned Poisson fits of stream + background, including the survey footprint, give the track, width and density.
- Control strips calibrate the segment significance; the measured Pal 5 track agrees with the published one.